Hybrid Retrieval (TF-IDF/BM25 + Dense).


# 1 Cấu hình đường dẫn

In [6]:
import os

CHUNKS_PATH = r"D:\GitHub\ChatBot\output_nghidinh\chunks_clean_norm.json"
GOLD_PATH   = r"D:\GitHub\ChatBot\retrieval\gold_200_natural.jsonl"   # hoặc gold_200_strong.jsonl
FAISS_PATH  = r"D:\GitHub\ChatBot\vector_data\legal_hf_cosine\index.faiss"

MODEL_NAME  = "Quockhanh05/Vietnam_legal_embeddings"

CAND_N = 100   # số ứng viên TF-IDF
TOPK   = 10    # topK sau rerank

MATCH_MODE = "dieu_khoan"  # "dieu" hoặc "dieu_khoan"

# 2 Load chunks + goldset

In [7]:
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

chunks = load_json(CHUNKS_PATH)
gold   = load_jsonl(GOLD_PATH)

len(chunks), len(gold)

(1861, 200)

# 3 Build TF-IDF index (candidate retriever)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [c["text"] for c in chunks]

tfidf = TfidfVectorizer(
    lowercase=True,
    analyzer="word",
    ngram_range=(1,2),  # bắt cụm từ pháp lý
    min_df=2,
    max_df=0.95
)

X = tfidf.fit_transform(texts)  # sparse matrix (N_chunks, V)
X.shape

(1861, 9869)

# 4 Load Dense (FAISS + SentenceTransformer)

In [9]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

faiss_index = faiss.read_index(FAISS_PATH)
encoder = SentenceTransformer(MODEL_NAME)

print("FAISS ntotal:", faiss_index.ntotal)

FAISS ntotal: 1861


# 5 Hàm match ground-truth & metric

In [10]:
def norm_str(x):
    return "" if x is None else str(x).strip()

def match(gt, meta, mode="dieu_khoan"):
    # đúng theo van_ban + dieu (+ khoan nếu mode dieu_khoan)
    if norm_str(gt.get("van_ban")) != norm_str(meta.get("van_ban")):
        return False
    if norm_str(gt.get("dieu")) != norm_str(meta.get("dieu")):
        return False
    if mode == "dieu":
        return True
    return norm_str(gt.get("khoan")) == norm_str(meta.get("khoan"))

def mrr_from_ranks(ranks):
    s = 0.0
    for r in ranks:
        if r > 0:
            s += 1.0 / r
    return s / len(ranks) if ranks else 0.0

# 6 Hàm Hybrid Search (TF-IDF → Dense rerank)

In [11]:
def hybrid_search(query: str, cand_n=100, topk=10):
    # 1) TF-IDF candidate
    q_t = tfidf.transform([query])  # (1, V)
    scores = (X @ q_t.T).toarray().ravel()  # (N,)

    cand_n = min(cand_n, len(scores))
    cand_ids = np.argpartition(-scores, cand_n)[:cand_n]
    cand_ids = cand_ids[np.argsort(-scores[cand_ids])]  # sort desc

    # 2) Dense rerank inside candidates
    qv = encoder.encode([query], normalize_embeddings=True).astype("float32")[0]

    # reconstruct vector của từng candidate từ FAISS (IndexFlatIP ok)
    cand_vecs = np.vstack([faiss_index.reconstruct(int(i)) for i in cand_ids]).astype("float32")

    dense_scores = cand_vecs @ qv  # cosine(dot) because normalized
    rerank_ids = cand_ids[np.argsort(-dense_scores)]

    return rerank_ids[:topk].tolist()

# 7 Evaluate Hybrid (Recall@K, MRR)

In [12]:
Ks = [1, 3, 5, 10]
Ks = [k for k in Ks if k <= TOPK]

hit_at = {K: 0 for K in Ks}
ranks = []

for g in gold:
    q = g["question"]

    top_ids = hybrid_search(q, cand_n=CAND_N, topk=TOPK)

    found = 0
    for r, cid in enumerate(top_ids, start=1):
        meta = chunks[cid].get("metadata", {})
        if match(g, meta, mode=MATCH_MODE):
            found = r
            break

    ranks.append(found)
    for K in Ks:
        if found > 0 and found <= K:
            hit_at[K] += 1

N = len(gold)
result = {
    "N": N,
    "cand": CAND_N,
    "TopK": TOPK,
    "match_mode": MATCH_MODE,
    **{f"Recall@{K}": hit_at[K]/N for K in Ks},
    "MRR": mrr_from_ranks(ranks)
}

result

{'N': 200,
 'cand': 100,
 'TopK': 10,
 'match_mode': 'dieu_khoan',
 'Recall@1': 0.075,
 'Recall@3': 0.15,
 'Recall@5': 0.205,
 'Recall@10': 0.34,
 'MRR': 0.13838492063492064}

In [13]:
fail_shown = 0

for g in gold:
    q = g["question"]
    top_ids = hybrid_search(q, cand_n=CAND_N, topk=TOPK)

    found = any(match(g, chunks[cid].get("metadata", {}), mode=MATCH_MODE) for cid in top_ids)
    if found:
        continue

    print("Q:", q)
    print("GT:", g["van_ban"], "Điều", g["dieu"], "Khoản", g["khoan"])
    print("Top3:")
    for cid in top_ids[:3]:
        m = chunks[cid]["metadata"]
        print(" -", m.get("van_ban","")[:80], "| dieu=", m.get("dieu"), "khoan=", m.get("khoan"))
    print("-"*80)

    fail_shown += 1
    if fail_shown >= 5:
        break

Q: Nguyên tắc liên quan đến hành, chính, giáo là gì?
GT: NGHỊ ĐỊNH Giớ: ĐẾN Ngày 16.1.6.2025 Quy định về phân quyền, phân cấp trong lĩnh vực quản lý nhà nước của Bộ Giáo dục và Đào tạo Điều 12 Khoản 6
Top3:
 - NGHỊ ĐỊNH Giớ: ĐẾN Ngày 16.1.6.2025 Quy định về phân quyền, phân cấp trong lĩnh  | dieu= 13 khoan= 
 - NGHỊ ĐỊNH Giớ: ĐẾN Ngày 16.1.6.2025 Quy định về phân quyền, phân cấp trong lĩnh  | dieu= 12 khoan= 4
 - NGHỊ ĐỊNH Ngày.13.1.6.2025 Quy định về phân quyền, phân cấp; phân định thẩm quyề | dieu= 5 khoan= 1
--------------------------------------------------------------------------------
Q: Quy định về mốc 22 tháng liên quan đến luật, tượng, phòng là gì?
GT: NGHỊ ĐỊNH Quy định phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Nông nghiệp và Môi trường Điều 33 Khoản 1
Top3:
 - NGHỊ ĐỊNH Ngày 13.16.12025 Quy định về phân quyền, phân cấp trong lĩnh vực đối n | dieu= 22 khoan= 3
 - NGHỊ ĐỊNH Quy định phân định thẩm quyền của chính quyền địa ph